# 02 — Ortholog Mapping

Map mouse gene symbols to human orthologs using the Ensembl REST API and a curated fallback table.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.config import *
from src.data_extraction import load_all_data
from src.ortholog_mapping import map_orthologs, mapping_summary, HARDCODED_ORTHOLOGS

In [ ]:
data = load_all_data()
sig = data['sig_proteins']
mouse_genes = sig['mouse_gene'].tolist()
print(f'Genes to map: {len(mouse_genes)}')

In [ ]:
# Map without API (fast, offline-safe)
cache_path = EXTERNAL_DIR / 'human_orthologs_cache.csv'
result_no_api = map_orthologs(mouse_genes, use_api=False, cache_path=cache_path)
print(f'Mapped (offline): {len(result_no_api)}')
result_no_api.head(10)

In [ ]:
mapping_summary(result_no_api)

In [ ]:
# Optional: run with Ensembl API (requires internet)
# result_api = map_orthologs(mouse_genes, use_api=True, cache_path=cache_path)
# mapping_summary(result_api)
print('(Ensembl API call commented out — set use_api=True to enable)')

In [ ]:
# Save
result_no_api.to_csv(PROCESSED_DIR / 'human_orthologs.csv', index=False)
print('Saved to data/processed/human_orthologs.csv')
result_no_api.head()

In [ ]:
# Check key proteins
for gene in ['Aplp1', 'Chl1', 'Mapt', 'App', 'Sort1']:
    row = result_no_api[result_no_api['mouse_symbol'].str.lower() == gene.lower()]
    if len(row):
        print(f'  {gene} -> {row.iloc[0]["human_symbol"]}  ({row.iloc[0]["source"]})')
    else:
        print(f'  {gene} -> NOT MAPPED')